# `00_sync_repo` — sync GitHub → Terra workspace

Run **on the Terra Workbench VM** (pmi-ops browser login). No laptop `gsutil`.

## What this notebook does

1. **Checkout** latest `kvg/aou-lr-phase-2` (disposable clone; local edits inside it are discarded)
2. **Stage scripts** → `$WORKSPACE_BUCKET/scripts/`
3. **Stage notebooks** → `$WORKSPACE_BUCKET/notebooks/` and `./notebooks/` beside the clone (`00_sync_repo.ipynb` itself is **not** restaged)
4. **Stage WDLs** → `$WORKSPACE_BUCKET/wdl/…` (plus `bam_to_contig/regions/` BEDs) and print a **manual import checklist** for any that changed vs the bucket
5. **Upsert** `flare_lai_exp` from `flare/configs/lai_exp.tsv`
6. **Optional:** Methods-repo snapshot (usually **403** on AoU billing-project namespace — leave off)
7. **Optional:** submit incomplete `flare_lai_exp` rows (`SUBMIT_FLARE=True`)

Keep this notebook in workspace `edit/` (not inside the clone).


## Config


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("AOU_LR_REPO_URL", "https://github.com/kvg/aou-lr-phase-2.git")
REF = os.environ.get("AOU_LR_REF", "main")
CLONE_DIR = Path(os.environ.get("AOU_LR_REPO_DIR", str(Path.cwd() / "aou-lr-phase-2")))
NOTEBOOK_DEST = Path(os.environ.get("AOU_LR_NOTEBOOK_DEST", str(Path.cwd() / "notebooks")))

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""

UPSERT_TABLES = ["flare_lai_exp"]

# AoU billing-project Methods namespace is usually not writable (HTTP 403).
UPDATE_METHODS = []
METHOD_NAMESPACE = os.environ.get("TERRA_METHOD_NAMESPACE", "")
BUMP_CONFIGS = []

SUBMIT_FLARE = os.environ.get("AOU_LR_SUBMIT_FLARE", "").lower() in {"1", "true", "yes"}
SUBMIT_ENTITY_IDS = None  # e.g. ["pin_gen8_afr_amr"]
SUBMIT_ONLY_INCOMPLETE = True
SUBMIT_CONFIG_NAME = "FlareByPopulation"

DRY_RUN = os.environ.get("AOU_LR_SYNC_DRY_RUN", "").lower() in {"1", "true", "yes"}

print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET", ""))
print("CLONE_DIR:", CLONE_DIR)
print("NOTEBOOK_DEST:", NOTEBOOK_DEST)
print("REF:", REF)
print("UPDATE_METHODS:", UPDATE_METHODS)
print("SUBMIT_FLARE:", SUBMIT_FLARE)
print("DRY_RUN:", DRY_RUN)


## Bootstrap

Clone/pull the disposable repo mirror and put `scripts/` on `sys.path`.


In [ ]:
import subprocess
import sys

def _run(cmd, **kw):
    print("+", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd)
    return subprocess.run(cmd, check=True, text=True, **kw)

token = GITHUB_TOKEN
url = REPO_URL
auth_url = (
    f"https://x-access-token:{token}@" + url.split("https://", 1)[1]
    if token and url.startswith("https://")
    else url
)

CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CLONE_DIR.exists() and any(CLONE_DIR.iterdir()) and not (CLONE_DIR / ".git").is_dir():
    raise SystemExit(f"{CLONE_DIR} exists and is not a git checkout")

if (CLONE_DIR / ".git").is_dir():
    _run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
    _run(["git", "fetch", "--tags", "--force", "origin"], cwd=str(CLONE_DIR))
    _run(["git", "reset", "--hard", f"origin/{REF}"], cwd=str(CLONE_DIR))
    _run(["git", "clean", "-fd"], cwd=str(CLONE_DIR))
    _run(["git", "checkout", "-B", REF, f"origin/{REF}"], cwd=str(CLONE_DIR))
else:
    try:
        _run(["git", "clone", "--branch", REF, "--single-branch", auth_url, str(CLONE_DIR)])
    except subprocess.CalledProcessError:
        _run(["git", "clone", auth_url, str(CLONE_DIR)])
        _run(["git", "checkout", REF], cwd=str(CLONE_DIR))
if token:
    _run(["git", "remote", "set-url", "origin", REPO_URL], cwd=str(CLONE_DIR))

scripts = CLONE_DIR / "scripts"
assert (scripts / "terra_sync_repo.py").is_file(), scripts
if str(scripts) not in sys.path:
    sys.path.insert(0, str(scripts))
print("import path:", scripts)
print("HEAD:", _run(["git", "rev-parse", "--short", "HEAD"], cwd=str(CLONE_DIR), capture_output=True).stdout.strip())


## Sync + optional submit


In [ ]:
import json
from terra_sync_repo import sync_all

if not os.environ.get("WORKSPACE_BUCKET") and not DRY_RUN:
    raise SystemExit("WORKSPACE_BUCKET unset — run on a Terra notebook VM (or DRY_RUN=true)")

report = sync_all(
    repo_url=REPO_URL,
    ref=REF,
    clone_dir=CLONE_DIR,
    github_token=GITHUB_TOKEN or None,
    notebook_dest=NOTEBOOK_DEST,
    stage_scripts_flag=True,
    stage_notebooks_flag=True,
    stage_wdls_flag=True,
    upsert_tables=UPSERT_TABLES,
    update_methods=UPDATE_METHODS or None,
    method_namespace=METHOD_NAMESPACE or None,
    bump_configs=BUMP_CONFIGS or None,
    submit_flare=SUBMIT_FLARE,
    submit_entity_ids=SUBMIT_ENTITY_IDS,
    submit_only_incomplete=SUBMIT_ONLY_INCOMPLETE,
    submit_config_name=SUBMIT_CONFIG_NAME,
    dry_run=DRY_RUN,
)
print(json.dumps(report.to_dict(), indent=2))

need = report.wdls_manual_import or []
print("\n=== WDL manual import checklist ===")
if need:
    print(f"{len(need)} WDL(s) new or changed vs bucket — re-import in Terra → Workflows:")
    for uri in need:
        print(f"  • {uri}")
else:
    print("None — staged WDLs match existing bucket copies.")

if report.warnings:
    print("\nWarnings:")
    for w in report.warnings:
        print("-", w)


## After sync

1. Check the **WDL manual import checklist** printed above. For each GCS URI listed: Terra → Workflows → import/replace that WDL (AoU Methods-repo Create is usually 403). New: `BamToContig.wdl` plus `$WORKSPACE_BUCKET/bam_to_contig/regions/`.
2. Confirm `$WORKSPACE_BUCKET/scripts/` and `$WORKSPACE_BUCKET/notebooks/` look fresh.
3. Confirm `flare_lai_exp` matches `flare/configs/lai_exp.tsv`.
4. To launch incomplete rows: `SUBMIT_FLARE = True` and re-run sync (or set `SUBMIT_ENTITY_IDS`).
5. Score finished rows with `flare_02_lai_exp_compare.ipynb` (Part 8).
6. Haplotig extraction: import `sample-hifi-hg38-all-cohorts` once (not auto-upserted), smoke-test one row, then Julie's four jobs — `bam_to_contig/README.md`.
